In [ ]:
from __future__ import annotations

import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers


def residual_block(x, filters: int, stride: int = 1, drop: float = 0.0):
    shortcut = x

    x = layers.Conv2D(
        filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4),
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(
        filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(1e-4),
    )(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(1e-4),
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    if drop > 0:
        x = layers.Dropout(drop)(x)
    return x


def build_avoidance_model(input_shape=(72, 128, 3), num_actions: int = 3) -> tf.keras.Model:
    """
    Compact residual CNN for indoor obstacle-avoidance steering.
    Output classes:
      0 = left
      1 = forward
      2 = right
    """
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 255.0)(inputs)

    x = layers.Conv2D(
        32,
        kernel_size=5,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding="same")(x)

    x = residual_block(x, 32, stride=1, drop=0.05)
    x = residual_block(x, 64, stride=2, drop=0.10)
    x = residual_block(x, 64, stride=1, drop=0.10)
    x = residual_block(x, 128, stride=2, drop=0.15)
    x = residual_block(x, 128, stride=1, drop=0.15)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)

    outputs = layers.Dense(num_actions, activation="softmax", name="action")(x)

    model = Model(inputs=inputs, outputs=outputs, name="indoor_obstacle_avoidance_cnn")
    return model


def compile_model(model: tf.keras.Model, lr: float = 1e-3) -> tf.keras.Model:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model
